# Embedding methane plumes into an AVIRIS-4 flight line

This tutorial walks through one embedding run on an AVIRIS-4 flight line.
We look at how SPECTRA reads the L1 radiance file and its PARGE metadata
file into its common `SensorScene`, embed two large-eddy-simulation (LES)
methane plumes, and then check the output file: where the plumes landed,
that the attenuation follows the absorption table, that the plume mass is
conserved, and that nothing else in the file changed.

You need:

- an AVIRIS-4 L1 radiance file in sensor geometry (`<scene>_rdn.nc`) and
  its metadata file (`<scene>_rdn_metadata.nc`);
- a `PlumeColumn` file of an LES plume (here a 290 kg/h MicroHH release on
  a 1 km x 1 km domain at 1 m resolution);
- the libRadtran/ARTS resource files of the forward model.

The output is a byte copy of the radiance file in which only the plume
pixels are attenuated, with the embedding metadata added to its root group.
The flight line used here (about 10 GB) was flown over Pau on the day of
the controlled release that the LES plume simulates. The paths below are
placeholders: replace them with your files.

In [ ]:
from pathlib import Path

import dask
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import xarray as xr
import yaml

import spectra
from spectra.contracts import convert_column_units
from spectra.plume.io import open_plume_column
from spectra.sensors import get_sensor
from spectra.sensors.native import open_native_radiance, split_variable, to_scene_dims
from spectra.testing import run_adapter_conformance

SCENE = "M016_240918_FRA_Pau_Methane_9000ft_Line_10001_105554_000"
RADIANCE = Path(f"/path/to/{SCENE}_rdn.nc")
METADATA = Path(f"/path/to/{SCENE}_rdn_metadata.nc")
PLUME = Path("/path/to/plume.plumecolumn.nc")
RESOURCES = Path("/path/to/data-unit-abs-spec")
OUTPUT = Path(f"/path/to/out/{SCENE}_rdn_embedded.nc")
FILES = {"radiance": RADIANCE, "metadata": METADATA}

## 1. The native files

An AVIRIS-4 flight line comes as two NetCDF files in sensor geometry. The
helper below lists every variable of a file with its dimensions, type,
units and fill value.

In [ ]:
def describe(path: Path) -> None:
    """Print every variable of a NetCDF file with dims, type, units and fill."""

    def walk(group: netCDF4.Dataset, prefix: str) -> None:
        for name, variable in group.variables.items():
            units = getattr(variable, "units", "")
            fill = getattr(variable, "_FillValue", "")
            dims = ", ".join(variable.dimensions)
            print(f"  {prefix}{name}({dims}) {variable.dtype} [{units}] fill={fill}")
        for name, child in group.groups.items():
            walk(child, f"{prefix}{name}/")

    with netCDF4.Dataset(path) as dataset:
        sizes = {name: len(dim) for name, dim in dataset.dimensions.items()}
        print(path.name, sizes)
        walk(dataset, "")


describe(RADIANCE)
describe(METADATA)

The radiance file holds `radiance(across, along, band)` in W m-2 nm-1 sr-1
with NaN as fill value, the band centres and widths as the data variables
`wavelength` and `fwhm`, and a scalar acquisition `time`. The metadata file
from the PARGE orthorectification holds the per-pixel geometry on the same
`(across, along)` grid: projected `x`/`y`, `lon`/`lat`, the surface
`elevation`, the signed `scan_zenith` (degrees x 100), the view
`absolute_azimuth` (degrees x 10) and the `aircraft_altitude`. The solar
angles of the flight line are attributes of `cast_shadow`, and the UTM zone
is the `epsg` attribute of `x` and `y`:

In [ ]:
with netCDF4.Dataset(METADATA) as dataset:
    for name in ("cast_shadow", "x"):
        variable = dataset[name]
        print(name, {key: variable.getncattr(key) for key in variable.ncattrs()})

The radiance of the 2024/2025 campaigns is stored contiguously with the
bands of a pixel next to each other; newer converter output is chunked
and stores the time in epoch seconds. Both are read.

In [ ]:
with netCDF4.Dataset(RADIANCE) as dataset:
    variable = dataset["radiance"]
    print(variable.dimensions, variable.shape)
    print("chunks:", variable.chunking(), "filters:", variable.filters())

The `aviris4` adapter knows these files. `ROLES` names the files of a scene
(both are required), and `LAYOUT` says where the radiance lives and how its
native dimensions map to SPECTRA's `(across, along, band)`. SPECTRA reads
through this layout and later writes the attenuated pixels back through it,
so import and export use the same mapping. For AVIRIS-4 the native names
already are the common ones.

In [ ]:
adapter = get_sensor("aviris4")
print("roles:", dict(adapter.ROLES))
print("radiance:", adapter.LAYOUT.role, "file, variable", adapter.LAYOUT.variable)
print("dimensions:", adapter.LAYOUT.dims)

## 2. Ingesting the scene

`open_scene` turns the two files into a `SensorScene`, lazily: the
radiance and the metadata are dask arrays, and the radiance keeps its
native units. The adapter takes `x`/`y` and the CRS from the metadata file,
the solar zenith and azimuth from `cast_shadow` (scalars for the flight
line), VZA as |`scan_zenith`| / 100, the view azimuth as
`absolute_azimuth` / 10, the sensor altitude from `aircraft_altitude` and
the surface elevation from `elevation`. The pixel area comes from the
stored pixel sizes when the file has them, otherwise from the centre
spacing. The `time` of the two files must agree.

`spectra.embed` opens the scene itself. We open it here only to look at it
and to choose where the plumes go.

In [ ]:
scene = adapter.open_scene(FILES)
scene

In [ ]:
radiance = scene["radiance"]
print(type(radiance.data).__name__, radiance.dims, radiance.shape)
print("units:", radiance.attrs["units"])
print(scene.attrs["scene_id"], scene.attrs["acquisition_time"], scene.attrs["crs"])

The panels show the scene in sensor geometry (every 4th pixel): the
radiance in the 2300 nm methane window and the geometry the forward model
will use.

In [ ]:
band = int(np.abs(scene["wavelength"] - 2300.0).argmin())
panels = {
    f"radiance {float(scene['wavelength'][band]):.0f} nm": radiance.isel(band=band),
    "surface elevation (m)": scene["surface_elevation"],
    "sensor altitude (m)": scene["sensor_altitude"],
    "VZA (deg)": scene["vza"],
    "pixel area (m2)": scene["pixel_area"],
}
fig, axes = plt.subplots(1, len(panels), figsize=(18, 4.5), sharey=True)
for ax, (title, field) in zip(axes, panels.items(), strict=True):
    field.astype(float)[::4, ::4].plot.imshow(
        ax=ax, x="across", y="along", robust=True, add_labels=False
    )
    ax.set_title(title)
plt.tight_layout()
print(f"valid pixels: {float(scene['valid'].mean()):.1%}")

A spectrum from the middle of the scene. The forward model covers
1000-2500 nm (the range of its optical-depth files), and only bands inside
it can be attenuated.

In [ ]:
centre = {"across": scene.sizes["across"] // 2, "along": scene.sizes["along"] // 2}
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(scene["wavelength"], radiance.isel(centre), lw=1)
ax.axvspan(1000, 2500, color="0.9", zorder=0, label="forward-model range")
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel(f"radiance ({radiance.attrs['units']})")
ax.legend();

`run_adapter_conformance` checks the opened scene against the
`SensorScene` contract. It also compares the radiance spectra of randomly
sampled pixels with a direct read of the native file through `LAYOUT`, so
the scene SPECTRA works on is the radiance in the file, value for value.

In [ ]:
run_adapter_conformance(adapter, FILES)
print("the scene follows the contract and matches the native file")

## 3. The plume

A `PlumeColumn` file holds the vertical column (kg m-2) of one LES tracer
on the model grid `(time, y, x)`, the emission rate and the position of the
emission source. One snapshot (`time_index`) is embedded per plume.

In [ ]:
plume = open_plume_column(PLUME)
TIME_INDEX = 120
snapshot = plume.isel(time=TIME_INDEX)
cell_area = float(plume.attrs["dx"]) * float(plume.attrs["dy"])
print(plume.attrs["case_id"], plume.attrs["gas"], dict(plume.sizes))
print(
    f"snapshot {TIME_INDEX} at {snapshot['time'].values}: "
    f"{float(snapshot['emission_rate']) * 3600:.0f} kg/h, "
    f"{float(snapshot['column'].sum()) * cell_area:.1f} kg in the domain"
)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
snapshot["column"].plot.imshow(ax=ax, cmap="magma", robust=True)
ax.plot(plume.attrs["source_x"], plume.attrs["source_y"], "c*", ms=12)
ax.set_title("LES column (kg m-2), emission source");

## 4. Describing the run

A run is one scene and any number of plumes. The forward model is the
libRadtran/ARTS model; every resource file is listed by role.

In [ ]:
forward_model = {
    "solar_reference": str(
        RESOURCES
        / "spectrum/hybrid_reference_spectrum_p005nm_resolution_c2021-03-04_with_unc.nc"
    ),
    "tau": {
        name: str(
            RESOURCES
            / f"spectrum/UVSPEC_moltau_midlatitude-summer_{name}_1000-2500nm-0.001nm.nc"
        )
        for name in ("all", "CH4")
    },
    "profiles": {
        name: str(RESOURCES / f"arts/midlatitude-summer/midlatitude-summer.{name}.xml")
        for name in ("z", "t", "CH4")
    },
    "backgrounds": {"CH4": 1.93},  # ppm
    "scenario": "midlatitude-summer",
    # Metre-scale pixels resolve strong plume cores.
    "enhancement_max_ppm": 2000,
}

We embed two plumes. The first has its emission source pinned to a
geographic position, taken here from a pixel in the middle of the scene,
and is rotated by 30 degrees. The second goes to a random pixel (drawn from
the run seed) where the whole plume lies inside the swath. Anchors can also
be pixels, and reflection and emission scaling are available too: see
`docs/run-file.md`.

In [ ]:
pixel = {
    "across": int(0.5 * scene.sizes["across"]),
    "along": int(0.4 * scene.sizes["along"]),
}
lon = float(scene["lon"].isel(pixel))
lat = float(scene["lat"].isel(pixel))
run = {
    "sensor": "aviris4",
    "files": {role: str(path) for role, path in FILES.items()},
    "output": str(OUTPUT),
    "overwrite": True,
    "seed": 2026,
    "forward_model": forward_model,
    "plumes": [
        {
            "file": str(PLUME),
            "time_index": TIME_INDEX,
            "anchor": {"lon": lon, "lat": lat},
            "rotation": 30,
        },
        {"file": str(PLUME), "time_index": 150, "anchor": "random"},
    ],
}
print(yaml.safe_dump(run["plumes"], sort_keys=False))

## 5. Embedding

`spectra.embed` places each plume and regrids it onto the pixel grid with
exact polygon overlap, which conserves mass. Plumes whose windows overlap
form a cluster. Each cluster gets one absorption table from the forward
model at its median geometry; this takes about a minute and a half per
cluster. Finally the native file is copied and the plume pixels are
attenuated (Lambert-Beer).

In [ ]:
outcome = spectra.embed(run)
pd.DataFrame(
    [
        {
            "plume": source.index,
            "anchor": source.anchor_mode,
            "across": source.anchor_across,
            "along": source.anchor_along,
            "lon": source.lon,
            "lat": source.lat,
            "cluster": source.cluster,
            "injected mass (kg)": source.injected_mass_kg,
            "outside fraction": source.outside_fraction,
        }
        for source in outcome.result.sources
    ]
).set_index("plume")

## 6. Checking the output

We read the original and the embedded radiance the same way, through the
adapter's layout on `(across, along, band)`, and the injected column of
each plume from the root group of the output. The column is stored as
`injected_ch4_mass(source, across, along)` on the native dimensions;
`to_scene_dims` puts them in the common order.

In [ ]:
def read_radiance(path: Path) -> xr.DataArray:
    """Open the radiance of a granule on (across, along, band), lazily."""
    data = open_native_radiance(path, adapter.LAYOUT, chunks=adapter.chunks)
    return data.assign_coords(
        across=np.arange(data.sizes["across"]),
        along=np.arange(data.sizes["along"]),
        band=np.arange(data.sizes["band"]),
    )


before = read_radiance(RADIANCE)
after = read_radiance(OUTPUT)
embedded = xr.open_dataset(OUTPUT)
column = to_scene_dims(embedded["injected_ch4_mass"], adapter.LAYOUT)
column = column.assign_coords(across=before["across"], along=before["along"])
wavelength = scene["wavelength"].values

# The band where methane absorbs most, from the first cluster's table.
first = outcome.result.clusters[0]
strong = first.bands.start + int(
    first.lut["optical_depth"].isel(enhancement=-1).argmin()
)
print(f"strongest absorption at {wavelength[strong]:.1f} nm (band {strong})")


def plume_box(field: xr.DataArray, margin: int = 10) -> dict[str, slice]:
    """Return the index box of the pixels above 1 % of the peak, plus a margin."""
    above = (field > 0.01 * float(field.max())).transpose("across", "along").values
    across = np.flatnonzero(above.any(axis=1))
    along = np.flatnonzero(above.any(axis=0))
    return {
        "across": slice(max(across[0] - margin, 0), across[-1] + 1 + margin),
        "along": slice(max(along[0] - margin, 0), along[-1] + 1 + margin),
    }

### The plumes in the radiance

For each plume, zoomed onto the pixels above 1 % of its peak column: the
original and the embedded radiance at the strongest band (same colour
scale), their ratio, and the injected column. Even at this band the plume
absorbs at most a few tens of percent, which can be hard to see against
the surface. Their ratio, the transmittance, shows the plume clearly:
below one where it absorbs, exactly one elsewhere, with the shape of the
injected column.

In [ ]:
for source in outcome.result.sources:
    box = plume_box(column.isel(source=source.index))
    old = before.isel(box).isel(band=strong).load()
    new = after.isel(box).isel(band=strong).load()
    low, high = np.nanpercentile(old, [2, 98])
    image = {"x": "across", "y": "along", "add_labels": False}
    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
    old.plot.imshow(ax=axes[0], cmap="gray", vmin=low, vmax=high, **image)
    new.plot.imshow(ax=axes[1], cmap="gray", vmin=low, vmax=high, **image)
    (new / old).plot.imshow(ax=axes[2], cmap="viridis", vmax=1, **image)
    column.isel(box).isel(source=source.index).plot.imshow(
        ax=axes[3], cmap="magma", **image
    )
    titles = ("original", "embedded", "embedded / original", "injected column (kg m-2)")
    for ax, title in zip(axes, titles, strict=True):
        ax.set_title(f"plume {source.index}: {title}")
    plt.tight_layout()

### Placement on the ground

The first plume was anchored at a longitude and latitude. Drawing the
transmittance on the pixels' geographic coordinates shows the plume
starting at the requested position (the cross), which is also the
emission source recorded in the file (`source_lon`, `source_lat`).

In [ ]:
box = plume_box(column.isel(source=0))
ratio = (after / before).isel(box).isel(band=strong)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
mesh = ax.pcolormesh(
    scene["lon"].isel(box),
    scene["lat"].isel(box),
    ratio,
    cmap="viridis",
    vmax=1,
    shading="gouraud",
)
fig.colorbar(mesh, label="embedded / original")
ax.plot(lon, lat, "r+", ms=18, mew=2, label="requested anchor")
ax.plot(
    float(embedded["source_lon"][0]),
    float(embedded["source_lat"][0]),
    "co",
    mfc="none",
    ms=12,
    mew=1.5,
    label="source_lon, source_lat",
)
ax.ticklabel_format(useOffset=False)
ax.xaxis.set_major_locator(plt.MaxNLocator(4))
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.legend(loc="lower right");

### The spectrum at the plume peak

At the pixel with the largest column the embedded radiance dips in the
methane bands around 2200-2400 nm (top, zoomed), and more weakly near
1650 nm. Outside the attenuated range recorded in the file the ratio
(bottom) is exactly one.

In [ ]:
peak = column.sum("source").argmax(...)
old = before.isel(peak).values
new = after.isel(peak).values
low = embedded.attrs["spectra_attenuated_wavelength_min_nm"]
high = embedded.attrs["spectra_attenuated_wavelength_max_nm"]
zoom = (wavelength > 2150) & (wavelength < 2450)
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 6))
top.plot(wavelength[zoom], old[zoom], ".-", lw=1, label="original")
top.plot(wavelength[zoom], new[zoom], ".-", lw=1, label="embedded")
top.set_xlabel("wavelength (nm)")
top.set_ylabel(radiance.attrs["units"])
top.legend()
bottom.plot(wavelength, new / old, lw=1, color="k")
bottom.axvspan(low, high, color="0.9", zorder=0, label="attenuated range")
bottom.set_xlabel("wavelength (nm)")
bottom.set_ylabel("embedded / original")
bottom.legend()
plt.tight_layout()
print(f"outside {low:.0f}-{high:.0f} nm the ratio is one:")
outside = (wavelength < low) | (wavelength > high)
print(bool(np.all((new == old)[outside] | np.isnan(old[outside]))))

### Lambert-Beer against the absorption table

Each cluster's absorption table gives the optical depth of a vertical
column in every band. The transmittance of a pixel is
exp(optical depth), interpolated at the pixel's column. So -ln(ratio) at
the strongest band, plotted against the column in ppm-m, must lie on the
table's curve (red: the table nodes, linear in between). Only pixels whose
attenuation exceeds the float32 resolution of the radiance (1e-6) are
shown.

In [ ]:
for cluster in outcome.result.clusters:
    window = cluster.window.indexers
    total = column.isel(window).sum("source").assign_attrs(units="kg m-2")
    amount = convert_column_units(total, "ppm-m", gas="ch4").values
    depth = -np.log((after / before).isel(window).isel(band=strong).values)
    resolved = (amount > 0) & (depth > 1e-6)
    table = cluster.lut["optical_depth"].isel(wavelength=strong - cluster.bands.start)
    nodes = cluster.lut["vertical_amount"].values
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(amount[resolved], depth[resolved], ".", ms=3, label="pixels")
    ax.loglog(nodes, -table.values, "r-o", ms=4, lw=1, label="absorption table")
    ax.set_xlabel("vertical column (ppm-m)")
    ax.set_ylabel("-ln(embedded / original)")
    ax.set_title(f"cluster {cluster.index}, {wavelength[strong]:.1f} nm")
    ax.legend()

### Mass conservation

The regridding conserves mass. The LES snapshot mass times the emission
scale and the fraction inside the swath must equal the mass on the sensor
grid stored in `source_injected_mass`. Summing the stored column times the
scene's pixel area gives nearly the same mass from the output file alone.
The small difference is there because `pixel_area` is estimated from the
centre spacing, while the regridder uses the exact pixel polygons.

In [ ]:
rows = []
for spec, source in zip(outcome.config.plumes, outcome.result.sources, strict=True):
    snapshot_mass = float(plume["column"].isel(time=spec.time_index).sum()) * cell_area
    les = snapshot_mass * spec.emission_scale
    on_grid = float(embedded["source_injected_mass"][source.index])
    from_file = float((column.isel(source=source.index) * scene["pixel_area"]).sum())
    rows.append(
        {
            "plume": source.index,
            "LES mass (kg)": les,
            "outside fraction": source.outside_fraction,
            "LES mass inside swath (kg)": les * (1 - source.outside_fraction),
            "source_injected_mass (kg)": on_grid,
            "column x pixel_area (kg)": from_file,
        }
    )
pd.DataFrame(rows).set_index("plume")

### Nothing else changed

One pass over both files finds every pixel and every band whose radiance
differs. Changed pixels must lie inside the injected plumes. Changed bands
must lie inside the attenuated range. Pixels with a vanishingly thin
column can keep their value, because the attenuation there is below
float32 resolution.

In [ ]:
differs = (after != before) & before.notnull()
changed_pixels, changed_bands = dask.compute(
    differs.any("band"), differs.any(["across", "along"])
)
plume_pixels = column.sum("source") > 0
print(
    f"pixels changed: {int(changed_pixels.sum())} "
    f"(of {int(plume_pixels.sum())} with a positive column)"
)
print("pixels changed outside the plumes:", int((changed_pixels & ~plume_pixels).sum()))
changed = wavelength[changed_bands.values]
print(
    f"bands changed: {changed.size}, {changed.min():.1f}-{changed.max():.1f} nm "
    f"(attenuated range {low:.1f}-{high:.1f} nm)"
)

Every other variable (`wavelength`, `fwhm`, `time` and the index
coordinates) is unchanged, and so are the original attributes. The only
additions are in the root group: the `source` dimension, the injected
column, the `source_*` variables and the `spectra_*` attributes. The
metadata file is not copied; it applies to the output as it is.

In [ ]:
radiance_group, radiance_name = split_variable(adapter.LAYOUT.variable)
original_tree = xr.open_datatree(RADIANCE)
embedded_tree = xr.open_datatree(OUTPUT)
for node in original_tree.subtree:
    old = node.to_dataset(inherit=False)
    new = embedded_tree[node.path].to_dataset(inherit=False)
    in_group = node.path.strip("/") == (radiance_group or "")
    skip = {radiance_name} if in_group else set()
    same = [
        old[name].identical(new[name]) for name in old.variables if name not in skip
    ]
    attrs = all(np.array_equal(new.attrs.get(key), old.attrs[key]) for key in old.attrs)
    print(f"{node.path:26s} {len(same):2d} variables same: {all(same)}, attrs: {attrs}")
added = sorted(
    set(embedded_tree.to_dataset(inherit=False).variables)
    - set(original_tree.to_dataset(inherit=False).variables)
)
print("added to the root group:", added)

## 7. Embedding metadata and reproducibility

Per-plume provenance is stored on the `source` dimension, run provenance
in `spectra_*` attributes.

In [ ]:
sources = embedded[[name for name in embedded.data_vars if name.startswith("source_")]]
sources.to_dataframe().T

In [ ]:
pd.Series(
    {
        key: value
        for key, value in embedded.attrs.items()
        if key.startswith("spectra_") and key != "spectra_run_config"
    }
).to_frame("value")

`spectra_run_config` is the resolved run file: the seed is set and the
random anchor is filled in with the pixel that was drawn. Running it again
reproduces the output. Saved as YAML, it also runs from the command line:
`spectra embed run.yaml --dry-run` prints the resolved run without running
the forward model, `spectra embed run.yaml` writes the output.

In [ ]:
resolved = yaml.safe_load(embedded.attrs["spectra_run_config"])
print(yaml.safe_dump(resolved["plumes"], sort_keys=False))
run_file = OUTPUT.with_suffix(".yaml")
run_file.write_text(embedded.attrs["spectra_run_config"])
print(spectra.load_run(run_file).plumes[1].anchor)

## Notes

- The output is a regular AVIRIS-4 radiance file: open it with the tools
  you use for the original, together with the original metadata file.
- Outputs are never overwritten unless `overwrite` is set, and an embedded
  file is refused as input, so plumes are never embedded twice.
- A plume core stronger than the absorption table stops the run before the
  forward model runs, and the message names the `enhancement_max_ppm` it
  needs.
- On NFSv3 file systems, set `HDF5_USE_FILE_LOCKING=FALSE` if opening the
  output copy for writing fails.